# Notebook 03 — Knowledge Base Artifacts

Produces two JSON knowledge-base files:

1. `data/processed/career_interest_mapping.json` — built **exclusively from Dataset 1**
   (`AI-based Career Recommendation System.csv`, ~200 rows).  
   Maps each career to a normalised weight distribution over interests.

2. `data/processed/career_skills.json` — built **exclusively from Dataset 2**
   (`data/processed/career_training.csv`, ~5 000 rows).  
   Maps each career to its required-skill list (frequency-threshold filtered).

The two datasets are **never concatenated** at any point.

---
## 0. Path setup — add project root to `sys.path`

In [ ]:
import sys
import os

# Notebook lives in AI-Career-Recommendation/notebooks/
# Project root (containing utils/) is one level up.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print('Project root added to sys.path:', project_root)

---
## Part A — `career_interest_mapping.json` (Dataset 1 only)

### A1. Load Dataset 1

In [ ]:
import pandas as pd

ds1_path = '../data/raw/AI-based Career Recommendation System.csv'
df1 = pd.read_csv(ds1_path)

print('=== Dataset 1 loaded ===')
print(f'Shape: {df1.shape}')
print(f'Columns: {df1.columns.tolist()}')
print()
print('Null counts:')
print(df1.isnull().sum())

### A2. Drop PII and unused columns

In [ ]:
cols_to_drop = ['CandidateID', 'Name', 'Age', 'Recommendation_Score']
existing_to_drop = [c for c in cols_to_drop if c in df1.columns]

df1 = df1.drop(columns=existing_to_drop)
print(f'Dropped: {existing_to_drop}')
print(f'Remaining columns: {df1.columns.tolist()}')

### A3. Import normalization functions

In [ ]:
from utils.preprocessing import normalize_career, split_semicolon_field

print('normalize_career and split_semicolon_field imported.')

### A4. Normalize career labels and split interests

In [ ]:
df1['Recommended_Career'] = df1['Recommended_Career'].astype(str).apply(normalize_career)
df1['Interests'] = df1['Interests'].astype(str).apply(split_semicolon_field)

print(f'Unique normalized careers: {df1["Recommended_Career"].nunique()}')
print('Career distribution:')
print(df1['Recommended_Career'].value_counts().to_string())

### A5. Count co-occurrences (career, interest)

In [ ]:
from collections import defaultdict

counts = defaultdict(lambda: defaultdict(int))

for _, row in df1.iterrows():
    career = row['Recommended_Career']
    for interest in row['Interests']:
        counts[career][interest] += 1

print(f'Careers with co-occurrence data: {len(counts)}')

# Show sample — top 3 interests for the first career
sample_career = next(iter(counts))
sample_interests = sorted(counts[sample_career].items(), key=lambda x: -x[1])[:3]
print(f'Sample — {sample_career}: {sample_interests}')

### A6. Normalise counts to weights

In [ ]:
career_interest_map = {}

for career, interest_counts in counts.items():
    total = sum(interest_counts.values())
    career_interest_map[career] = {
        interest: count / total
        for interest, count in interest_counts.items()
    }

# Spot-check: weights for one career should sum to ~1.0
sample_sum = sum(career_interest_map[sample_career].values())
print(f'Weight sum for "{sample_career}": {sample_sum:.8f}  (expected ~1.0)')

### A7. Flag careers with fewer than 3 rows in `_metadata`

In [ ]:
career_row_counts = df1['Recommended_Career'].value_counts()
low_sample_careers = career_row_counts[career_row_counts < 3].index.tolist()

print(f'Careers with < 3 rows: {low_sample_careers}')

career_interest_map['_metadata'] = {
    'low_sample_careers': low_sample_careers
}

### A8. Save `career_interest_mapping.json`

In [ ]:
import json

output_path_interest = '../data/processed/career_interest_mapping.json'

with open(output_path_interest, 'w', encoding='utf-8') as f:
    json.dump(career_interest_map, f, indent=2, ensure_ascii=False)

# Count data careers (exclude _metadata key)
data_careers = [k for k in career_interest_map if k != '_metadata']
print(f'Saved {len(data_careers)} careers to: {output_path_interest}')

---
## Part B — `career_skills.json` (Dataset 2 / career_training.csv only)

### B1. Load `career_training.csv`

In [ ]:
training_path = '../data/processed/career_training.csv'
df2 = pd.read_csv(training_path)

print('=== career_training.csv loaded ===')
print(f'Shape: {df2.shape}')
print(f'Columns: {df2.columns.tolist()}')
print()
print('Career distribution:')
print(df2['Recommended Career'].value_counts().to_string())

### B2. Count skill occurrences per career

In [ ]:
career_skill_counts = defaultdict(lambda: defaultdict(int))
career_row_totals = defaultdict(int)

for _, row in df2.iterrows():
    career = row['Recommended Career']
    skills = [s.strip() for s in str(row['Skills']).split(';') if s.strip()]
    career_row_totals[career] += 1
    for skill in skills:
        career_skill_counts[career][skill] += 1

print(f'Unique careers in training data: {len(career_skill_counts)}')

# Sample
sample_career2 = next(iter(career_skill_counts))
top3 = sorted(career_skill_counts[sample_career2].items(), key=lambda x: -x[1])[:3]
print(f'Sample — {sample_career2} ({career_row_totals[sample_career2]} rows): top skills = {top3}')

### B3. Apply frequency threshold and build `career_skills.json`

In [ ]:
career_skills = {}

for career, skill_counts in career_skill_counts.items():
    record_count = career_row_totals[career]
    threshold = max(2, record_count * 0.10)
    qualified_skills = sorted(
        skill for skill, count in skill_counts.items() if count >= threshold
    )
    career_skills[career] = qualified_skills

# Summary
total_skill_entries = sum(len(v) for v in career_skills.values())
print(f'Careers: {len(career_skills)},  total skill entries: {total_skill_entries}')
print(f'Sample — {sample_career2}: {career_skills[sample_career2]}')

### B4. Save `career_skills.json`

In [ ]:
output_path_skills = '../data/processed/career_skills.json'

with open(output_path_skills, 'w', encoding='utf-8') as f:
    json.dump(career_skills, f, indent=2, ensure_ascii=False)

print(f'Saved {len(career_skills)} careers to: {output_path_skills}')

---
## Part C — Validation

### C1. Validate `career_interest_mapping.json`

In [ ]:
with open(output_path_interest, 'r', encoding='utf-8') as f:
    loaded_interest_map = json.load(f)

# Separate data careers from metadata
data_careers_loaded = {k: v for k, v in loaded_interest_map.items() if k != '_metadata'}
metadata = loaded_interest_map.get('_metadata', {})

print(f'Careers in map (excluding _metadata): {len(data_careers_loaded)}')
print(f'Low-sample careers: {metadata.get("low_sample_careers", [])}')
print()

In [ ]:
# Assert weight sum ≈ 1.0 for every career
weight_errors = []
for career, weights in data_careers_loaded.items():
    total = sum(weights.values())
    if abs(total - 1.0) >= 1e-6:
        weight_errors.append((career, total))

assert len(weight_errors) == 0, f'Weight sum != 1.0 for careers: {weight_errors}'
print(f'[PASS] All {len(data_careers_loaded)} careers have interest weights summing to ~1.0')

In [ ]:
# Assert no fabricated interests — all interests must come from the raw Dataset 1
all_raw_interests = set()
for interests_list in df1['Interests']:
    all_raw_interests.update(interests_list)

fabricated = set()
for career, weights in data_careers_loaded.items():
    for interest in weights:
        if interest not in all_raw_interests:
            fabricated.add(interest)

assert len(fabricated) == 0, f'Fabricated interests found: {fabricated}'
print(f'[PASS] No fabricated interests — all {len(all_raw_interests)} unique interests come from Dataset 1')

### C2. Validate `career_skills.json`

In [ ]:
with open(output_path_skills, 'r', encoding='utf-8') as f:
    loaded_career_skills = json.load(f)

print(f'Careers in career_skills.json: {len(loaded_career_skills)}')

In [ ]:
# Assert all careers from career_training.csv are present
training_careers = set(df2['Recommended Career'].unique())
missing_from_kb = training_careers - set(loaded_career_skills.keys())

assert len(missing_from_kb) == 0, f'Careers missing from career_skills.json: {missing_from_kb}'
print(f'[PASS] All {len(training_careers)} training careers are present in career_skills.json')

In [ ]:
# Assert no empty skill lists for careers with >= 10 rows
career_counts_df = df2['Recommended Career'].value_counts()
well_represented = career_counts_df[career_counts_df >= 10].index.tolist()

empty_skill_careers = [
    c for c in well_represented
    if c in loaded_career_skills and len(loaded_career_skills[c]) == 0
]

assert len(empty_skill_careers) == 0, f'Careers with >= 10 rows but empty skills: {empty_skill_careers}'
print(f'[PASS] No empty skill lists for the {len(well_represented)} careers with >= 10 rows')

In [ ]:
print()
print('=== All validations passed ===')
print(f'career_interest_mapping.json: {len(data_careers_loaded)} careers')
print(f'career_skills.json:           {len(loaded_career_skills)} careers')
print(f'Low-sample careers flagged:   {metadata.get("low_sample_careers", [])}')